In [1]:
# @title
from google.colab import drive
drive.mount('/content/drive')

import os
import sys

PROJECT_DIR = "/content/drive/MyDrive/DumLLM"

os.chdir(PROJECT_DIR)

sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

print("DumLLM initialized!")
print("Working directory:", os.getcwd())
print("Project files:", os.listdir(PROJECT_DIR))

Mounted at /content/drive
DumLLM initialized!
Working directory: /content/drive/MyDrive/DumLLM
Project files: ['requirements.txt', '.venv', '.git', 'notebooks', 'public', '.gitignore', 'start_DumLLM.ipynb']


## 3.3 Attending to different parts of the input with self-attention

### 3.3.1 A simple self-attention mechanism without trainable weights

In [3]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [6]:
input_query = inputs[1] # (x^2)
input_1 = inputs[0]

In [27]:
res = 0.
i = 0
for idx,el in enumerate(inputs[i]):
  res += inputs[i][idx] * input_query[idx]

print(res)

tensor(0.9544)


In [23]:
i = 0

res = torch.dot(inputs[i],input_query)
res

tensor(0.9544)

In [25]:
# dot product for all tokens with query
query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0])
for i,x_i in enumerate(inputs):
  attn_scores_2[i] = torch.dot(x_i,query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [29]:
# normalizing with softmax
def softmax_naive(x):
  return torch.exp(x) / torch.exp(x).sum()

attn_scores_2_softmax = softmax_naive(attn_scores_2)
print(attn_scores_2_softmax)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [30]:
# more optimized version
attn_scores_2_softmax = torch.softmax(attn_scores_2,dim=0)
print(attn_scores_2_softmax)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [33]:
# weighted sum with the attention scores
query = inputs[1]

context_vec_2 = torch.zeros(query.shape)

for i,x_i in enumerate(inputs):
  context_vec_2 += attn_scores_2_softmax[i] * x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


### 3.3.2 Computing attention weights for all input tokens

In [36]:
attn_scores = torch.empty(inputs.shape[0], inputs.shape[0])

for i,x_i in enumerate(inputs):
  for j,x_j in enumerate(inputs):
    attn_scores[i, j] = torch.dot(x_i,x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [38]:
# matrix multiplication
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [39]:
# normaliszing
attn_weights = torch.softmax(attn_scores,dim=1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [41]:
# all at once
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores,dim=1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [42]:
# compute context vectors
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## 3.4 Implementing self-attention with trainable weights

### 3.4.1 Computing the attention weights step by step

In [48]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [46]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

In [58]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

In [59]:
print(query_2)
print(key_2)
print(value_2)

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)
tensor([0.4433, 1.1419], grad_fn=<SqueezeBackward4>)
tensor([0.3951, 1.0037], grad_fn=<SqueezeBackward4>)


In [70]:
keys = inputs @ W_key
values = inputs @ W_value

In [61]:
# dot product between Query and Key for Attention Score"
attn_scores_22 = torch.dot(query_2,key_2)
print(attn_scores_22)

tensor(1.8524, grad_fn=<DotBackward0>)


In [62]:
attn_scores_2 = query_2 @ keys.T
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440],
       grad_fn=<SqueezeBackward4>)


In [67]:
# normalizing as per the paper
d_k = keys.shape[1]

attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [71]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)
